Si consideri il dataset clinico SUPPORT2, contenente informazioni demografiche, fisiologiche e di laboratorio relative a pazienti ospedalizzati.

In [ ]:
import pandas as pd
df = pd.read_csv("./dataset_esercitazione.csv",sep=',')
print(df)

Dopo aver caricato i dati, rimuovere le variabili di outcome e prognosi per 
evitare fenomeni di data leakage e selezionare una variabile target (ad esempio dzgroup).
Suddividere il dataset in training set (95%) e test set (5%) utilizzando un campionamento stratificato rispetto 
alla variabile target («death»), in modo da preservare la distribuzione delle classi.

In [2]:
from sklearn.model_selection import train_test_split

X=df.drop(columns=['dzgroup','dzclass'])
y=df['death']
X_tr, X_te, y_tr, y_te=train_test_split(X,y,test_size=0.05,stratify=df['death']) 

Gestire i dati mancanti applicando imputazione con mediana per le variabili numeriche e con un valore 
costante (ad esempio "Unknown") per quelle categoriche; successivamente codificare le variabili categoriche tramite Ordinal Encoding

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

numeric_features = X_tr.select_dtypes(include=['number']).columns.tolist()
categorical_features = [col for col in X_tr.columns if col not in numeric_features]

report_mancanti = (X_tr.isnull().sum() / len(X_tr)) * 100

# Usiamo Pandas per estrarre direttamente una lista con i nomi delle sole colonne da imputare
colonne_con_nan = report_mancanti[report_mancanti > 0].index.tolist()

print(f"Colonne totali con dati mancanti: {len(colonne_con_nan)}")

# Dividiamo queste colonne "problematiche" in numeriche e categoriche
nan_numeriche = [col for col in colonne_con_nan if col in numeric_features]
nan_categoriche = [col for col in colonne_con_nan if col in categorical_features]


# Imputiamo SOLO le colonne numeriche che ne hanno bisogno
if len(nan_numeriche) > 0:
    imputer_num = SimpleImputer(strategy='median')
    imputer_num.set_output(transform='pandas')
    
    # Applichiamo solo al sotto-dataset delle colonne bucate
    X_tr[nan_numeriche] = imputer_num.fit_transform(X_tr[nan_numeriche])
    X_te[nan_numeriche] = imputer_num.transform(X_te[nan_numeriche])

# Imputiamo SOLO le colonne categoriche che ne hanno bisogno
if len(nan_categoriche) > 0:
    imputer_cat = SimpleImputer(strategy='constant', fill_value='Unknown')
    imputer_cat.set_output(transform='pandas')
    
    # Applichiamo solo al sotto-dataset delle colonne bucate
    X_tr[nan_categoriche] = imputer_cat.fit_transform(X_tr[nan_categoriche])
    X_te[nan_categoriche] = imputer_cat.transform(X_te[nan_categoriche])

# ==========================================
# 4. ENCODING
# ==========================================
# L'encoding invece va fatto sempre su TUTTE le categoriche, che avessero NaN o meno
if len(categorical_features) > 0:
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-18)
    encoder.set_output(transform='pandas')
    
    X_tr[categorical_features] = encoder.fit_transform(X_tr[categorical_features])
    X_te[categorical_features] = encoder.transform(X_te[categorical_features])


Applicare lo scaling delle feature numeriche tramite z-scoring, quindi calcolare e analizzare la matrice di correlazione, individuando eventuali coppie di variabili altamente correlate e discutendo la presenza di multicollinearità.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
import matplotlib.pyplot as plt
import numpy as np

scaler = StandardScaler()
scaler.set_output(transform='pandas') # Fondamentale per mantenere X_tr come DataFrame

# Fit e Transform su X_tr, SOLO Transform su X_te per le feature numeriche
X_tr[numeric_features] = scaler.fit_transform(X_tr[numeric_features])
X_te[numeric_features] = scaler.transform(X_te[numeric_features])

# Calcolo Matrice di Correlazione sull'intero X_tr (numeriche + categoriche encodate)
matrice_corr = X_tr.corr()

# --- Plot Matrice ---
fig, ax = plt.subplots(figsize=(14, 12))
cax = ax.imshow(matrice_corr, cmap='coolwarm', vmin=-1, vmax=1)
fig.colorbar(cax, fraction=0.046, pad=0.04)

ax.set_xticks(np.arange(len(matrice_corr.columns)))
ax.set_yticks(np.arange(len(matrice_corr.columns)))
ax.set_xticklabels(matrice_corr.columns, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(matrice_corr.columns, fontsize=9)

plt.title("Matrice di Correlazione (Training Set Scalato)", fontsize=14)
plt.tight_layout()
plt.show()

# --- Analisi Multicollinearità ---
soglia_allarme = 0.75 
print("\n--- Analisi Multicollinearità ---")
corr_pairs = matrice_corr.unstack().sort_values(key=abs, ascending=False)
corr_pairs = corr_pairs[corr_pairs != 1.0]
coppie_pericolose = corr_pairs[abs(corr_pairs) > soglia_allarme]

feature_da_rimuovere = set() # Usiamo un set per non avere duplicati

if coppie_pericolose.empty:
    print("Nessun problema grave di multicollinearità.")
else:
    print(f"ATTENZIONE: Trovate variabili con correlazione > {soglia_allarme}:")
    for index, valore in coppie_pericolose[::2].items():
        var1, var2 = index
        print(f" - {var1} e {var2}: correlazione di {valore:.3f}")
        # Scegliamo arbitrariamente di rimuovere la seconda variabile della coppia per risolvere la collinearità
        feature_da_rimuovere.add(var2)

# ==========================================
# PUNTO 5: GESTIONE MULTICOLLINEARITÀ E FEATURE SELECTION
# ==========================================
if feature_da_rimuovere:
    print(f"\nRimuovo le seguenti feature ridondanti per gestire la multicollinearità: {list(feature_da_rimuovere)}")
    X_tr = X_tr.drop(columns=list(feature_da_rimuovere))
    X_te = X_te.drop(columns=list(feature_da_rimuovere))

print("\n--- Selezione delle Feature Più Rilevanti ---")
# Usiamo SelectKBest per tenere solo le 10 feature più utili rispetto al target
k_features = 10
# Attenzione: Sostituisci 'y_tr' con la tua vera variabile target (es. il vettore che contiene la colonna 'death' che hai separato al punto 1)
selector = SelectKBest(score_func=f_classif, k=k_features)

# Addestriamo il selettore
selector.fit(X_tr, y_tr)

# Stampiamo le feature che hanno vinto
feature_selezionate = X_tr.columns[selector.get_support()].tolist()
print(f"Le {k_features} feature più importanti sono: {feature_selezionate}")

# Trasformiamo definitivamente i dataset
X_tr_final = selector.transform(X_tr)
X_te_final = selector.transform(X_te)

Gestire la multicollinearità mediante rimozione di feature ridondanti o tramite una trasformazione lineare (ad esempio PCA), e infine applicare un metodo di selezione delle feature per identificare le variabili più rilevanti, commentando criticamente i risultati ottenuti